In [1]:
# ============================================================
# 1. MONTAR GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# ============================================================
# 2. IMPORTAR LIBRERÍAS
# ============================================================

import pandas as pd
import os

In [3]:
# ============================================================
# 3. RUTA Y CARGA DEL FICHERO
# ============================================================

ruta = "/content/drive/MyDrive/Visualización/turismo_interno_prov_ccaa.csv"

df = pd.read_csv(
  ruta,
  sep=";",
  encoding="latin-1"
)

df.head()

,AÑO,MES,CCAA_ORIGEN,PROVINCIA_ORIGEN,CCAA_DESTINO,PROVINCIA_DESTINO,TURISTAS,PERNOCTACIONES,ESTANCIA_MEDIA
0,2024,1,Andalucía,Almería,Andalucía,Cádiz,"4737,0","26292,0","5,6"
1,2024,1,Andalucía,Almería,Andalucía,Córdoba,"4196,0","24586,0","5,9"
2,2024,1,Andalucía,Almería,Andalucía,Granada,"50708,0","176829,0","3,5"
3,2024,1,Andalucía,Almería,Andalucía,Huelva,"2003,0","13339,0","6,7"
4,2024,1,Andalucía,Almería,Andalucía,Jaén,"11126,0","66282,0","6,0"


In [4]:
# ============================================================
# 4. CONTENIDO
# ============================================================

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62485 entries, 0 to 62484
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   AÑO                62485 non-null  int64 
 1   MES                62485 non-null  int64 
 2   CCAA_ORIGEN        62485 non-null  object
 3   PROVINCIA_ORIGEN   62485 non-null  object
 4   CCAA_DESTINO       62485 non-null  object
 5   PROVINCIA_DESTINO  62485 non-null  object
 6   TURISTAS           62485 non-null  object
 7   PERNOCTACIONES     62485 non-null  object
 8   ESTANCIA_MEDIA     61926 non-null  object
dtypes: int64(2), object(7)
memory usage: 4.3+ MB


In [5]:
# ============================================================
# 5. DIMENSIONES DEL DATASET
# ============================================================

print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

Filas: 62485
Columnas: 9


In [6]:
from numpy import int64
# ============================================================
# 6. CONVERTIR VARIABLES NUMÉRICAS
# ============================================================

# Turistas y pernoctaciones → enteros
columnas_enteras = [
    "TURISTAS",
    "PERNOCTACIONES"
]

for columna in columnas_enteras:
    df[columna] = (
        df[columna]
        .str.replace(",", ".", regex=False)
        .astype(float)
        .astype("int64")
    )

# Estancia media → decimal
df["ESTANCIA_MEDIA"] = (
    df["ESTANCIA_MEDIA"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

df.dtypes

,0
AÑO,int64
MES,int64
CCAA_ORIGEN,object
PROVINCIA_ORIGEN,object
CCAA_DESTINO,object
PROVINCIA_DESTINO,object
TURISTAS,int64
PERNOCTACIONES,int64
ESTANCIA_MEDIA,float64


In [7]:
df.head()

,AÑO,MES,CCAA_ORIGEN,PROVINCIA_ORIGEN,CCAA_DESTINO,PROVINCIA_DESTINO,TURISTAS,PERNOCTACIONES,ESTANCIA_MEDIA
0,2024,1,Andalucía,Almería,Andalucía,Cádiz,4737,26292,5.6
1,2024,1,Andalucía,Almería,Andalucía,Córdoba,4196,24586,5.9
2,2024,1,Andalucía,Almería,Andalucía,Granada,50708,176829,3.5
3,2024,1,Andalucía,Almería,Andalucía,Huelva,2003,13339,6.7
4,2024,1,Andalucía,Almería,Andalucía,Jaén,11126,66282,6.0


In [8]:
# ============================================================
# 7. FILTRAR AÑO 2025
# ============================================================

df_2025 = df[df["AÑO"] == 2025].copy()


In [9]:
df_2025 = df[df["AÑO"] == 2025].copy()

In [10]:
# ============================================================
# 8. AGRUPAR POR ORIGEN Y DESTINO
# ============================================================

df_2025_agrupado = (
    df_2025
    .groupby(
        [
            "CCAA_ORIGEN",
            "PROVINCIA_ORIGEN",
            "CCAA_DESTINO",
            "PROVINCIA_DESTINO"
        ],
        as_index=False
    )
    .agg({
        "TURISTAS": "sum",
        "PERNOCTACIONES": "sum"
    })
)



In [11]:
# ============================================================
#  8. GUARDAR CSV PROCESADO
# ============================================================

fichero_final = "/content//drive/MyDrive/Visualización/TurismoInterno_2025.csv"

df_2025_agrupado.to_csv(
    fichero_final,
    index=False,
    encoding="utf-8-sig"
)

print("\nArchivo generado correctamente.")
print(fichero_final)


Archivo generado correctamente.
/content//drive/MyDrive/Visualización/TurismoInterno_2025.csv


In [12]:
# ============================================================
#  9. CREAR TABLA DE NODOS (POINTS)
#     TAMAÑO  = TURISTAS RECIBIDOS
# ============================================================

points = (
    df_2025_agrupado
    .groupby(
        ["CCAA_DESTINO", "PROVINCIA_DESTINO"],
        as_index=False
    )["TURISTAS"]
    .sum()
)

points = points.rename(
    columns={
        "PROVINCIA_DESTINO": "id",
        "CCAA_DESTINO": "group",
        "TURISTAS": "size"
    }
)

points.head()


,group,id,size
0,Andalucía,Almería,5212154
1,Andalucía,Cádiz,12558970
2,Andalucía,Córdoba,4668886
3,Andalucía,Granada,8242542
4,Andalucía,Huelva,6709777


In [13]:
# ============================================================
#  9. CREAR TABLA DE NODOS (POINTS)
#     TAMAÑO  = TURISTAS RECIBIDOS
# ============================================================

fichero_points= "/content//drive/MyDrive/Visualización/TurismoInterno_2025_points.csv"

points.to_csv(
    fichero_points,
    index=False,
    encoding="utf-8-sig"
)

print("\nArchivo generado correctamente.")
print(fichero_points)



Archivo generado correctamente.
/content//drive/MyDrive/Visualización/TurismoInterno_2025_points.csv
